# Benchmark Runner — B1 + B4
**Conda env: `webagent-r1-unified`**  
**GPU: 2 and 3 (`CUDA_VISIBLE_DEVICES=2,3`)**

Run cells top to bottom. Each cell is independent — you can re-run just the model eval cell if needed.

In [16]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2,3"   # lock to GPU 2 & 3
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"  # suppress cosmetic warnings

import sys, json, re, time, torch
from pathlib import Path

FINETUNE_ROOT = Path("/data1/cs24mtech14020/home/TDL/NexusBots-TDL-Project/finetune")
sys.path.insert(0, str(FINETUNE_ROOT))

from config import ADAPTER_DIR, BASE_MODEL, MAX_SEQ_LENGTH, SYSTEM_PROMPT, TEST_PATH, RESULTS_DIR
print("CUDA devices visible:", torch.cuda.device_count())
print("Using:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

CUDA devices visible: 2
Using: ['NVIDIA RTX A6000', 'NVIDIA RTX A6000']


In [17]:
# ── Load test data ──────────────────────────────────────────────────────────
def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

test_rows = load_jsonl(TEST_PATH)
print(f"Loaded {len(test_rows)} test rows")

# For smoke test: set MAX_ROWS = 5. For full run: set MAX_ROWS = None
MAX_ROWS = None
if MAX_ROWS:
    test_rows = test_rows[:MAX_ROWS]
    print(f"[SMOKE] Limited to {MAX_ROWS} rows")

Loaded 100 test rows


In [18]:
# ── Load fine-tuned model (PEFT = base + LoRA adapter) ─────────────────────
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

adapter_path = str(ADAPTER_DIR)
base_model_id = BASE_MODEL

# Load tokenizer from base model ID — avoids transformers 4.57.x bug where
# Qwen3's extra_special_tokens list triggers an AttributeError on .keys()
print(f"Loading tokenizer from: {base_model_id}")
ft_tokenizer = AutoTokenizer.from_pretrained(base_model_id)

print(f"Loading base model: {base_model_id}")
_base = AutoModelForCausalLM.from_pretrained(
    base_model_id, dtype=torch.float16, device_map="cuda:0"
)
print("Merging LoRA adapter...")
ft_model = PeftModel.from_pretrained(_base, adapter_path)
ft_model.eval()
print(f"Fine-tuned model on: {next(ft_model.parameters()).device}")

Loading tokenizer from: Qwen/Qwen3-0.6B
Loading base model: Qwen/Qwen3-0.6B
Merging LoRA adapter...
Fine-tuned model on: cuda:0


In [19]:
# ── Load base model (no adapter, for comparison) ────────────────────────────
# Reuse ft_tokenizer — same vocab, no need to reload
base_tokenizer = ft_tokenizer

print(f"Loading base model: {BASE_MODEL}")
base_model_obj = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, dtype=torch.float16, device_map="cuda:0"
)
base_model_obj.eval()
print(f"Base model on: {next(base_model_obj.parameters()).device}")

Loading base model: Qwen/Qwen3-0.6B
Base model on: cuda:0


In [ ]:
# ── Helper functions ────────────────────────────────────────────────────────
import re as _re

def parse_json_from_text(text):
    text = text.strip()
    text = _re.sub(r"<think>[\s\S]*?</think>", "", text).strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    s = text.find("{")
    e = text.rfind("}") + 1
    if s >= 0 and e > s:
        try:
            return json.loads(text[s:e])
        except Exception:
            pass
    return {}

def infer(model, tokenizer, query, context_str):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Query: {query}\nContext: {context_str}"},
    ]
    # enable_thinking=False: training used full-conversation template which never
    # adds <think> to assistant messages. At inference, Qwen3 defaults to
    # enable_thinking=True which inserts <think> into the generation prompt,
    # causing the model to hallucinate ui_guide keys as tool names. Disabling
    # thinking matches the exact format the model was trained on.
    text = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=200, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    latency = (time.perf_counter() - t0) * 1000
    generated = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return parse_json_from_text(generated), latency

def extract_gold(row):
    for m in row.get("messages", []):
        if m["role"] == "assistant":
            obj = json.loads(m["content"])
            return obj["tool"], obj["arguments"], obj.get("ui_guide")
    return "", {}, None

def score_arg_f1(pred, gold):
    if not gold:
        return 1.0 if not pred else 0.0
    keys = set(gold) | set(pred)
    if not keys:
        return 1.0
    return sum(str(pred.get(k)).strip().lower() == str(gold.get(k)).strip().lower() for k in keys) / len(keys)

print("Helpers ready.")

In [21]:
# ── Run evaluation for one model ─────────────────────────────────────────────
from collections import defaultdict

def evaluate(label, model, tokenizer, rows):
    stats = defaultdict(lambda: {"ok": 0, "f1": 0.0, "n": 0, "lats": []})
    total_ok = 0
    n = len(rows)
    for i, row in enumerate(rows):
        gold_tool, gold_args, _ = extract_gold(row)
        lang = row.get("language", "en")
        user_msg = next(m["content"] for m in row["messages"] if m["role"] == "user")
        parts = user_msg.split("\nContext: ", 1)
        query = parts[0].replace("Query: ", "")
        ctx   = parts[1] if len(parts) > 1 else "{}"
        try:
            pred, lat = infer(model, tokenizer, query, ctx)
            pt = pred.get("tool", "")
            pa = pred.get("arguments", {})
        except Exception as e:
            pt, pa, lat = "", {}, 0.0
            print(f"  ERR row {i}: {e}")
        ok = int(pt == gold_tool)
        total_ok += ok
        stats[lang]["ok"] += ok
        stats[lang]["f1"] += score_arg_f1(pa, gold_args)
        stats[lang]["n"]  += 1
        stats[lang]["lats"].append(lat)
        bar = "#" * ((i+1)*30//n) + "-" * (30 - (i+1)*30//n)
        print(f"  [{bar}] {i+1:3d}/{n}  {'✓' if ok else '✗'}  pred={pt or '(none)':20s}  gold={gold_tool:20s}  run_acc={total_ok/(i+1):.2f}  lat={lat:.0f}ms", flush=True)
    print(f"\n  {label} DONE → tool_acc={total_ok/n:.4f}")
    return dict(stats)

print("evaluate() ready.")

evaluate() ready.


In [22]:
# ── SMOKE TEST — 1 row, confirm output is valid JSON ────────────────────────
smoke = evaluate("SMOKE", ft_model, ft_tokenizer, test_rows[:1])
print("Smoke test passed ✓" if smoke else "No output")

  [##############################]   1/1  ✗  pred=recommend             gold=search_products       run_acc=0.00  lat=14073ms

  SMOKE DONE → tool_acc=0.0000
Smoke test passed ✓


In [23]:
# ── DEBUG: inspect raw model output for 1 row ───────────────────────────────
row = test_rows[0]
gold_tool, gold_args, _ = extract_gold(row)
user_msg = next(m["content"] for m in row["messages"] if m["role"] == "user")
parts = user_msg.split("\nContext: ", 1)
query = parts[0].replace("Query: ", "")
ctx   = parts[1] if len(parts) > 1 else "{}"

msgs = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f"Query: {query}\nContext: {ctx}"},
]
text = ft_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = ft_tokenizer(text, return_tensors="pt").to(ft_model.device)
print(f"Input tokens: {inputs['input_ids'].shape[1]}")
print(f"Gold: tool={gold_tool}, args={gold_args}")

with torch.no_grad():
    out = ft_model.generate(**inputs, max_new_tokens=512, do_sample=False,
                            pad_token_id=ft_tokenizer.eos_token_id)

new_tokens = out[0][inputs["input_ids"].shape[1]:]
print(f"New tokens generated: {len(new_tokens)}")

# Show with special tokens to catch early-stop tokens
raw_with_special = ft_tokenizer.decode(new_tokens, skip_special_tokens=False)
print(f"\nRaw output (with special tokens):\n{repr(raw_with_special)}")

Input tokens: 1270
Gold: tool=search_products, args={'query': 'STEM education robot', 'category': 'Humanoid'}
New tokens generated: 222

Raw output (with special tokens):
'<think>\nOkay, the user is looking for an interactive learning robot. The context says the selected category is Humanoid. Since the query is about learning robots, I need to check if there\'s a humanoid robot that fits. The available humanoid robots are IDs 10-12: Miko 3, Wonder Workshop Dash, and LEGO Education Spike Prime. The user\'s query doesn\'t mention product IDs, so I should use the selectedCategory "Humanoid" and query the tools with that. No need for a tool call here. Wait, the rules say if the query has a category name, use that. Since the user specified "learning robot," maybe focus on that need. But the tool\'s category is Humanoid. So tool would be recommend with need="interactive learning robot", category="Humanoid". Let me make sure there\'s no other tool needed. No, that\'s the only tool matching th

In [24]:
# ── CELL 7: Fine-tuned model inference ──────────────────────────────────────
print("=== Qwen3-0.6B-FC (fine-tuned) ===")
ft_stats = evaluate("Qwen3-0.6B-FC", ft_model, ft_tokenizer, test_rows)

=== Qwen3-0.6B-FC (fine-tuned) ===
  [------------------------------]   1/100  ✗  pred=recommend             gold=search_products       run_acc=0.00  lat=13246ms
  [------------------------------]   2/100  ✓  pred=get_product           gold=get_product           run_acc=0.50  lat=7212ms
  [------------------------------]   3/100  ✓  pred=navigate_to           gold=navigate_to           run_acc=0.67  lat=8966ms
  [#-----------------------------]   4/100  ✓  pred=search_products       gold=search_products       run_acc=0.75  lat=6756ms
  [#-----------------------------]   5/100  ✓  pred=search_products       gold=search_products       run_acc=0.80  lat=5928ms
  [#-----------------------------]   6/100  ✗  pred=search_products       gold=get_product           run_acc=0.67  lat=8836ms
  [##----------------------------]   7/100  ✗  pred=recommend             gold=navigate_to           run_acc=0.57  lat=26442ms
  [##----------------------------]   8/100  ✗  pred=search_products       gold=re

In [25]:
# ── CELL 8: Base model inference ────────────────────────────────────────────
print("=== Qwen3-0.6B BASE ===")
base_stats = evaluate("Qwen3-0.6B-BASE", base_model_obj, base_tokenizer, test_rows)

=== Qwen3-0.6B BASE ===
  [------------------------------]   1/100  ✗  pred=recommend             gold=search_products       run_acc=0.00  lat=9398ms
  [------------------------------]   2/100  ✓  pred=get_product           gold=get_product           run_acc=0.50  lat=4468ms
  [------------------------------]   3/100  ✓  pred=navigate_to           gold=navigate_to           run_acc=0.67  lat=8678ms
  [#-----------------------------]   4/100  ✓  pred=search_products       gold=search_products       run_acc=0.75  lat=8584ms
  [#-----------------------------]   5/100  ✓  pred=search_products       gold=search_products       run_acc=0.80  lat=5563ms
  [#-----------------------------]   6/100  ✓  pred=get_product           gold=get_product           run_acc=0.83  lat=5001ms
  [##----------------------------]   7/100  ✓  pred=navigate_to           gold=navigate_to           run_acc=0.86  lat=10942ms
  [##----------------------------]   8/100  ✓  pred=recommend             gold=recommend     

In [26]:
# ── CELL 9: Print summary table & save results ───────────────────────────────
import csv

def summarise(label, stats):
    all_ok = sum(v["ok"] for v in stats.values())
    all_f1 = sum(v["f1"] for v in stats.values())
    all_n  = sum(v["n"]  for v in stats.values())
    all_lats = sorted(l for v in stats.values() for l in v["lats"])
    p50 = all_lats[len(all_lats)//2] if all_lats else 0
    return {
        "system": label,
        "tool_acc": round(all_ok/all_n, 4) if all_n else 0,
        "arg_f1":  round(all_f1/all_n, 4) if all_n else 0,
        "p50_ms":  round(p50, 1),
        "count":   all_n,
    }

rows_summary = [
    summarise("Qwen3-0.6B-FC (ours)", ft_stats),
    summarise("Qwen3-0.6B BASE",      base_stats),
]

print("\n=== FINAL RESULTS ===")
print(f"{'System':<25} {'Tool Acc':>9} {'Arg F1':>8} {'p50 (ms)':>10} {'N':>5}")
print("-" * 62)
for r in rows_summary:
    print(f"{r['system']:<25} {r['tool_acc']:>9.4f} {r['arg_f1']:>8.4f} {r['p50_ms']:>10.1f} {r['count']:>5}")

# Save CSV
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
csv_path = RESULTS_DIR / "b1_function_calling_nb.csv"
with open(csv_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["system","tool_acc","arg_f1","p50_ms","count"])
    w.writeheader()
    w.writerows(rows_summary)
print(f"\nSaved → {csv_path}")


=== FINAL RESULTS ===
System                     Tool Acc   Arg F1   p50 (ms)     N
--------------------------------------------------------------
Qwen3-0.6B-FC (ours)         0.4400   0.3283    12527.1   100
Qwen3-0.6B BASE              0.6500   0.4733     8800.8   100

Saved → /data1/cs24mtech14020/home/TDL/NexusBots-TDL-Project/research/results/b1_function_calling_nb.csv
